# Ejemplo de Clase: Visualización + Validación Cruzada + GridSearchCV

**Dataset:** Wine (UCI) – 178 muestras, 13 características químicas, 3 clases de vino.

**Objetivo de la clase:**  
1. Explorar y visualizar los datos (aplicar lo visto de percepción y elección de gráficos).  
2. Entender por qué una sola división Train/Test no es suficiente.  
3. Aplicar K-Fold y Stratified K-Fold.  
4. Encontrar los mejores hiperparámetros con `GridSearchCV`.

---
**Instrucciones para el docente:**  
Ejecutar celda por celda, preguntando a los alumnos qué esperan ver antes de cada visualización y antes de interpretar los resultados de CV y GridSearch.

## 0. Carga de librerías y datos

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.datasets import load_wine
from sklearn.model_selection import (
    train_test_split,
    cross_val_score,
    StratifiedKFold,
    KFold,
    GridSearchCV
)
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

# Estilo de gráficos
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["figure.figsize"] = (10, 5)
plt.rcParams["axes.titlesize"] = 14
plt.rcParams["axes.labelsize"] = 12

# Cargar dataset
wine = load_wine(as_frame=True)
X = wine.data
y = wine.target
df = X.copy()
df["target"] = y
df["clase"] = df["target"].map({0: "clase_0", 1: "clase_1", 2: "clase_2"})

print("Forma del dataset:", X.shape)
print("Clases:", wine.target_names)
print("\nDistribución de clases:")
print(df["clase"].value_counts().sort_index())
df.head()

## 1. Exploración visual (aplicación de los conceptos de la unidad)

Preguntas para la clase:
- ¿Qué tipo de gráfico usamos para comparar la distribución de una variable numérica por clase?
- ¿Qué principio Gestalt estamos aprovechando al colorear por clase?

In [ ]:
# 1.1 Distribución de clases (barras) – comparación de categorías
fig, ax = plt.subplots(figsize=(7, 4))
counts = df["clase"].value_counts().sort_index()
bars = ax.bar(counts.index, counts.values, color=["#A8864A", "#6B5A44", "#3A2C1A"], edgecolor="white")
ax.set_title("Distribución de clases en el dataset Wine")
ax.set_ylabel("Cantidad de muestras")
ax.set_xlabel("Clase de vino")
for bar, val in zip(bars, counts.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1, str(val),
            ha="center", va="bottom", fontweight="bold")
plt.tight_layout()
plt.show()

print("Observación: las clases están razonablemente balanceadas → K-Fold simple o Stratified funcionan bien.")

In [ ]:
# 1.2 Boxplots de características clave por clase (comparación de distribuciones)
features_interes = ["alcohol", "malic_acid", "flavanoids", "color_intensity"]

fig, axes = plt.subplots(2, 2, figsize=(12, 8))
axes = axes.ravel()

for i, feat in enumerate(features_interes):
    sns.boxplot(data=df, x="clase", y=feat, ax=axes[i], hue="clase", legend=False,
                palette=["#A8864A", "#6B5A44", "#3A2C1A"])
    axes[i].set_title(f"Distribución de {feat} por clase")
    axes[i].set_xlabel("")

plt.suptitle("Boxplots: comparación de características químicas entre clases", y=1.02, fontsize=14)
plt.tight_layout()
plt.show()

print("Pregunta a la clase: ¿Qué característica separa mejor las clases a simple vista?")

In [ ]:
# 1.3 Scatter plot (relación entre dos variables + clase)
fig, ax = plt.subplots(figsize=(8, 6))
sns.scatterplot(
    data=df, x="alcohol", y="flavanoids", hue="clase",
    palette=["#A8864A", "#6B5A44", "#3A2C1A"], s=70, alpha=0.8, ax=ax
)
ax.set_title("Relación Alcohol vs Flavonoides coloreada por clase")
ax.set_xlabel("Alcohol")
ax.set_ylabel("Flavonoides")
plt.legend(title="Clase")
plt.tight_layout()
plt.show()

print("Aquí aplicamos: forma visual (dispersión) + color como significado (clase) + principio de similitud (Gestalt).")

## 2. El problema de una sola partición Train/Test

Vamos a demostrar la **variabilidad del azar** haciendo varias divisiones aleatorias y midiendo el accuracy.

In [ ]:
model = RandomForestClassifier(n_estimators=50, max_depth=4, random_state=42)

accuracies = []
print("Accuracy en 10 divisiones aleatorias distintas (test_size=0.25):\n")
for i in range(10):
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.25, random_state=i, stratify=y
    )
    model.fit(X_train, y_train)
    acc = accuracy_score(y_test, model.predict(X_test))
    accuracies.append(acc)
    print(f"  Seed {i:2d} → Accuracy: {acc:.4f}")

print(f"\nPromedio: {np.mean(accuracies):.4f}")
print(f"Desviación estándar: {np.std(accuracies):.4f}")
print(f"Rango: {np.min(accuracies):.4f} – {np.max(accuracies):.4f}")

print("\n→ Conclusión: dependiendo de la semilla, el accuracy puede variar bastante.")
print("  Una sola división puede darnos una impresión demasiado optimista o pesimista.")

## 3. K-Fold y Stratified K-Fold

Ahora usamos validación cruzada para obtener una estimación más estable.

In [ ]:
# 3.1 K-Fold simple (sin estratificar)
kfold = KFold(n_splits=5, shuffle=True, random_state=42)
scores_kfold = cross_val_score(model, X, y, cv=kfold, scoring="accuracy")

print("=== K-Fold (5 folds) ===")
print("Scores por fold:", np.round(scores_kfold, 4))
print(f"Promedio: {scores_kfold.mean():.4f}  ±  {scores_kfold.std():.4f}")

In [ ]:
# 3.2 Stratified K-Fold (recomendado en clasificación)
skfold = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scores_skfold = cross_val_score(model, X, y, cv=skfold, scoring="accuracy")

print("=== Stratified K-Fold (5 folds) ===")
print("Scores por fold:", np.round(scores_skfold, 4))
print(f"Promedio: {scores_skfold.mean():.4f}  ±  {scores_skfold.std():.4f}")

print("\nObservación: al estar las clases balanceadas, la diferencia es pequeña.")
print("En datasets desbalanceados (ej. fraude) la diferencia suele ser mucho mayor.")

## 4. GridSearchCV – búsqueda de hiperparámetros

Definimos una rejilla pequeña para que se ejecute rápido en clase.

In [ ]:
param_grid = {
    "n_estimators": [30, 50, 100],
    "max_depth": [2, 4, 6, None],
    "min_samples_leaf": [1, 3]
}

print("Combinaciones a probar:", 
      len(param_grid["n_estimators"]) * 
      len(param_grid["max_depth"]) * 
      len(param_grid["min_samples_leaf"]))
print("(cada una se evalúa con 5-fold CV → total de entrenamientos = combinaciones × 5)")

In [ ]:
grid = GridSearchCV(
    estimator=RandomForestClassifier(random_state=42),
    param_grid=param_grid,
    cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=42),
    scoring="accuracy",
    n_jobs=-1,
    return_train_score=True,
    verbose=1
)

grid.fit(X, y)

print("\n=== Resultados de GridSearchCV ===")
print("Mejores hiperparámetros:", grid.best_params_)
print(f"Mejor score (CV): {grid.best_score_:.4f}")

In [ ]:
# Ver las 5 mejores combinaciones
results = pd.DataFrame(grid.cv_results_)
top5 = results[["params", "mean_test_score", "std_test_score", "mean_train_score"]].sort_values(
    "mean_test_score", ascending=False
).head(5)

print("Top 5 combinaciones:")
for i, row in top5.iterrows():
    print(f"  {row['params']} → test: {row['mean_test_score']:.4f} (±{row['std_test_score']:.4f}) | train: {row['mean_train_score']:.4f}")

## 5. Modelo final y evaluación honesta

Buena práctica:  
1. Separar un hold-out al inicio (aquí lo hacemos al final solo por didáctica).  
2. Usar GridSearch solo sobre el conjunto de entrenamiento.  
3. Evaluar **una sola vez** sobre el hold-out.

In [ ]:
# Separar hold-out (20%)
X_train, X_holdout, y_train, y_holdout = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

# GridSearch solo sobre train
grid_final = GridSearchCV(
    estimator=RandomForestClassifier(random_state=42),
    param_grid=param_grid,
    cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=42),
    scoring="accuracy",
    n_jobs=-1
)
grid_final.fit(X_train, y_train)

print("Mejores parámetros (sobre train):", grid_final.best_params_)
print(f"Mejor CV score (sobre train): {grid_final.best_score_:.4f}")

# Evaluación final en hold-out (solo una vez)
y_pred = grid_final.best_estimator_.predict(X_holdout)
print(f"\nAccuracy en hold-out: {accuracy_score(y_holdout, y_pred):.4f}")
print("\nClassification Report:")
print(classification_report(y_holdout, y_pred, target_names=wine.target_names))

In [ ]:
# Matriz de confusión
cm = confusion_matrix(y_holdout, y_pred)
fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt="d", cmap="YlOrBr", 
            xticklabels=wine.target_names, yticklabels=wine.target_names, ax=ax)
ax.set_xlabel("Predicho")
ax.set_ylabel("Real")
ax.set_title("Matriz de confusión – Hold-out")
plt.tight_layout()
plt.show()

## 6. Síntesis para la clase

| Concepto | Qué vimos |
|----------|-----------|
| **Visualización** | Barras (distribución), boxplots (comparación), scatter (relación + color = clase) |
| **Problema de una sola partición** | El accuracy varió según la semilla |
| **K-Fold / Stratified** | Estimación más estable del desempeño |
| **GridSearchCV** | Probamos 24 combinaciones × 5 folds y elegimos la mejor |
| **Hold-out final** | Evaluación honesta que no se usó durante la búsqueda |

**Mensaje final:**  
La visualización nos ayuda a entender los datos.  
La validación cruzada nos ayuda a no engañarnos con el desempeño.  
GridSearchCV nos ayuda a elegir hiperparámetros de forma sistemática.